# 01b — Limpieza e ingeniería

Dedup, descriptiva, imputación, temporalidad, festivos e índices finales (bloque 2 del cuaderno 01). Lee `firms_spain_full_enrichment.csv`; escribe `expanded/wildfire_data_cleaned.csv`.

## Imports (celda única del cuaderno)

In [ ]:
# Imports del cuaderno (celda única) — todas las dependencias aquí; las demás celdas no importan nada.
from IPython.display import display
import holidays
import numpy as np
import pandas as pd
from wildfire.config import load_config
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.dedup_cross_sensor import apply_modeling_schema, collapse_cross_sensor_dupes


## 2.1 — Eliminación de duplicados entre sensores

Colapsa detecciones duplicadas VIIRS/MODIS y aplica el esquema de modelado; produce `df_no_dupe`.

In [ ]:
# Eliminación de duplicados


df_no_dupe = apply_modeling_schema(
    collapse_cross_sensor_dupes(
        load_csv("data/processed/enriched/firms_spain_full_enrichment.csv")
    )
)


## 2.2 — Estadística descriptiva

Describe el `df_no_dupe` de la 2.1 (sin columnas de contexto); solo visualiza.

In [ ]:
# Estadística descriptiva


num_years = load_config()["ccaa"]["budget_lookback_years"]

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


DESCRIBE_DROP_COLUMNS = [
    "acq_date",
    "acq_time",
    "latitude",
    "longitude",
    "confidence_cat",
    "confidence_num",
    "daynight",
    "clc_class",
    "clc_class_N",
    "clc_class_S",
    "clc_class_W",
    "clc_class_E",
    "clc_class_NW",
    "clc_class_NE",
    "clc_class_SW",
    "clc_class_SE",
    "clc_uniform_surroundings",
    "ccaa",
    "type",
    "year",
    "is_day",
    "prevention_current",
    f"prevention_prev{num_years}",
    f"extinction_prev{num_years}"
]

df_prov_describe = df_no_dupe.drop(columns=DESCRIBE_DROP_COLUMNS)

print(df_prov_describe.columns)
display(df_prov_describe.describe())


## 2.3 — Limpieza de filas sin CCAA ni presupuestos

Elimina filas sin región o sin sumas (Ceuta/Melilla); produce `df_nan_clean`.

In [ ]:
# Limpieza de filas sin CCAA y sin presupuestos (Ceuta y Melilla)

display(df_no_dupe.isna().sum())

# Las filas sin ccaa o sin presupuestos (Ceuta/Melilla) se eliminan
# (ver recuentos de faltantes arriba).
df_nan_clean = df_no_dupe.dropna(subset=["ccaa", "prevention_current", f"prevention_prev{num_years}", f"extinction_prev{num_years}"])

## 2.4 — Imputación de `boundary_layer_height`

Rellena los faltantes con la mediana; el recuento se verifica en la salida.

In [ ]:
# Detección de valores nulos; asignación de valor a boundary_layer_height
# Los faltantes de boundary_layer_height (ver recuento arriba) se imputan
# con la mediana por ser una variable con atípicos.

df_nan_clean.fillna({"boundary_layer_height": df_nan_clean["boundary_layer_height"].median()}, inplace=True)

display(df_nan_clean.isna().sum())

print(df_nan_clean.shape)

## 2.5 — Detección de atípicos (IQR)

Regla de Tukey solo como informe; no elimina ni transforma nada.

In [ ]:
# Detección de outliers
# Detección valores atipicos / Outlier detection (main dataframe is df_nan_clean)

df_prov_describe = df_nan_clean.drop(columns=DESCRIBE_DROP_COLUMNS)

display(df_prov_describe.describe())

# We apply the Turkey IQR rule
Q1, Q3 = df_prov_describe.quantile(0.25), df_prov_describe.quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((df_prov_describe < Q1 - 1.5 * IQR) | (df_prov_describe > Q3 + 1.5 * IQR)).sum()
print(outlier_counts[outlier_counts > 0])

# Nuestros outliers son: frp, scan, brightness, brightness_ir, surface_pressure, wind_speed_10m, wind_gusts_10m, boundary_layer_height, vapour_pressure_deficit
# Our outliers are: frp, scan, brightness, brightness_ir, surface_pressure, wind_speed_10m, wind_gusts_10m, boundary_layer_height, vapour_pressure_deficit

# temperature_2m solo tiene 52 valores que se salen de los percentiles, así que no son outliers significativos
# temperature_2m only has 52 values that fall outside the percentiles, so they are not significant outliers.

# precipitation y rain no son outliers ya que la mayoría de los días no llueve (0), por lo que parecen outliers aunque no lo sean
# precipitation and rain are not outliers since most days it does not rain (0), so they seem like outliers even though they are not

## 2.6 — Descomposición temporal

`acq_date` a datetime más `month`, `day_of_year` y `es_fin_semana`.

In [ ]:
# DESCOMPOSICIÓN TEMPORAL
df_nan_clean['acq_date'] = pd.to_datetime(df_nan_clean['acq_date'])

df_nan_clean['month'] = df_nan_clean['acq_date'].dt.month
df_nan_clean['day_of_year'] = df_nan_clean['acq_date'].dt.dayofyear
df_nan_clean['es_fin_semana'] = df_nan_clean['acq_date'].dt.dayofweek.isin([5, 6]).astype(int)

## 2.7 — Tramos horarios de `acq_time`

Hora extraída de `acq_time` y recodificada en `franja_horaria`.

In [ ]:
# Segmentación de acq_time en tramos del día
df_nan_clean['acq_time_str'] = df_nan_clean['acq_time'].astype(str).str.zfill(4)
df_nan_clean['hora'] = pd.to_datetime(df_nan_clean['acq_time_str'], format='%H%M', errors='coerce').dt.hour

bins_hora = [-1, 6, 12, 18, 24]
labels_hora = ['Madrugada', 'Mañana', 'Tarde_Pico_Calor', 'Noche']
df_nan_clean['franja_horaria'] = pd.cut(df_nan_clean['hora'], bins=bins_hora, labels=labels_hora, right=False)

## 2.8 — Festivos y riesgo antrópico

Cruza fecha×CCAA con el calendario (`holidays.Spain`) en `es_festivo` y `riesgo_antropico_alto`.

In [ ]:
# Calcular festivos solo para combinaciones únicas de fecha/CCAA


# Festivos 
mapa_ccaa_holidays = {
    'Andalucía': 'AN', 'Aragón': 'AR', 'Asturias': 'AS', 'Islas Baleares': 'IB', 
    'Canarias': 'CN', 'Cantabria': 'CB', 'Castilla y León': 'CL', 
    'Castilla-La Mancha': 'CM', 'Cataluña': 'CT', 'Comunidad Valenciana': 'VC', 
    'Extremadura': 'EX', 'Galicia': 'GA', 'Madrid': 'MD', 
    'Región de Murcia': 'MC', 'Navarra': 'NC', 'País Vasco': 'PV', 'La Rioja': 'RI'
}

calendario_unico = df_nan_clean[['acq_date', 'ccaa']].dropna().drop_duplicates()

def evaluar_festivo(fila):
    codigo_region = mapa_ccaa_holidays.get(fila['ccaa'])
    if not codigo_region:
        return 0
    cal = holidays.Spain(years=fila['acq_date'].year, subdiv=codigo_region)
    return int(fila['acq_date'] in cal)

calendario_unico['es_festivo'] = calendario_unico.apply(evaluar_festivo, axis=1)

df_nan_clean = df_nan_clean.merge(calendario_unico, on=['acq_date', 'ccaa'], how='left')
df_nan_clean['es_festivo'] = df_nan_clean['es_festivo'].fillna(0).astype(int)

# Variable riesgo antrópico
df_nan_clean['riesgo_antropico_alto'] = ((df_nan_clean['es_fin_semana'] == 1) | (df_nan_clean['es_festivo'] == 1)).astype(int)


## 2.9 — Codificación cíclica

Seno/coseno de mes y día del año para la estacionalidad sin saltos.

In [ ]:
# Codificación cíclica con sin/cos

df_nan_clean['month_sin'] = np.sin(2 * np.pi * df_nan_clean['month'] / 12)
df_nan_clean['month_cos'] = np.cos(2 * np.pi * df_nan_clean['month'] / 12)
df_nan_clean['day_of_year_sin'] = np.sin(2 * np.pi * df_nan_clean['day_of_year'] / 365.25)
df_nan_clean['day_of_year_cos'] = np.cos(2 * np.pi * df_nan_clean['day_of_year'] / 365.25)


## 2.10 — Hectáreas MITECO e índices finales

Ratios por hectárea e índices meteorológicos derivados; escribe `expanded/wildfire_data_cleaned.csv`.

In [ ]:
# Hectáreas forestales (MITECO)

num_years = load_config()["ccaa"]["budget_lookback_years"]

superficie_forestal_ha = {
    'Andalucía': 4400000, 'Aragón': 2600000, 'Asturias': 770000,
    'Islas Baleares': 220000, 'Canarias': 570000, 'Cantabria': 360000,
    'Castilla y León': 5100000, 'Castilla-La Mancha': 3600000, 'Cataluña': 2000000,
    'Comunidad Valenciana': 1300000, 'Extremadura': 2700000, 'Galicia': 2000000,
    'Madrid': 430000, 'Región de Murcia': 510000, 'Navarra': 610000,
    'País Vasco': 490000, 'La Rioja': 340000
}

df_nan_clean['sup_forestal_ha'] = df_nan_clean['ccaa'].map(superficie_forestal_ha)

df_nan_clean['ratio_prevencion_extincion'] = df_nan_clean[f'prevention_prev{num_years}'] / (df_nan_clean[f'extinction_prev{num_years}'] + 1e-5)
df_nan_clean['euros_prevencion_por_ha'] = (df_nan_clean[f'prevention_prev{num_years}'] * 1000000) / df_nan_clean['sup_forestal_ha']
df_nan_clean['euros_extincion_por_ha'] = (df_nan_clean[f'extinction_prev{num_years}'] * 1000000) / df_nan_clean['sup_forestal_ha']


# Potencial de secado atmosférico acelerado por el viento
df_nan_clean['potencial_propagacion'] = df_nan_clean['vapour_pressure_deficit'] * df_nan_clean['wind_speed_10m']

df_nan_clean['indice_riesgo_meteo'] = df_nan_clean['potencial_propagacion'] / (df_nan_clean['soil_moisture_0_to_7cm'] + 0.01)

df_nan_clean['estres_termico_vegetacion'] = df_nan_clean['temperature_2m'] / (df_nan_clean['relative_humidity_2m'] + 1)


df_nan_clean['indice_riesgo_meteo_log'] = np.log1p(df_nan_clean['indice_riesgo_meteo'])

# 7. Limpieza final
columnas_basura = [
    'acq_time_str', 'hora',
    'es_fin_semana', 'es_festivo'
]
df_nan_clean = df_nan_clean.drop(columns=columnas_basura)

save_csv(df_nan_clean, "data/processed/expanded/wildfire_data_cleaned.csv")

display(df_nan_clean)
print(df_nan_clean.shape)


## 2.11 — Quemas agrícolas candidatas

El satélite no distingue quemas de rastrojo de incendios forestales (límite declarado en el plan no supervisado §2). Heurística reversible: **se apartan** las detecciones **dentro de la ventana** de quema controlada (15 oct–30 may, configurable por grupo de CCAA) **y** sobre combustibles herbáceos (`clc_class` 6 "Permanent herbaceous", 7 "Periodically herbaceous"): en meses fríos no hay régimen de incendio forestal sobre herbáceas, así que esas detecciones son casi con seguridad quemas gestionadas. Todo lo demás se conserva: en verano todo (cualquier clase), en invierno salvo herbáceas.

Parámetros editables arriba del código (grupos, ventanas, clases). La celda diagnostica antes de filtrar (conteos, FRP, mes) y falla si las sospechosas no tienen FRP menor que las conservadas. Escribe `expanded/wildfire_data_cleaned.csv` (conservadas) y `expanded/quemas_agricolas_candidatas.csv` (apartadas, para perfilar aparte).


In [ ]:
# Quemas agrícolas candidatas: dentro de ventana (15oct-30may) y sobre herbáceas (CLC 6/7)

GRUPOS_CCAA = {
    "norte": ["Galicia", "Asturias", "Cantabria", "País Vasco", "Navarra", "La Rioja"],
    "centro": ["Castilla y León", "Aragón", "Cataluña", "Madrid", "Castilla-La Mancha", "Extremadura"],
    "sur": ["Andalucía", "Comunidad Valenciana", "Región de Murcia", "Islas Baleares", "Canarias"],
}
VENTANAS = {"norte": ((10, 15), (5, 30)), "centro": ((10, 15), (5, 30)), "sur": ((10, 15), (5, 30))}
CLC_QUEMA = [6, 7]

grupo = df_nan_clean["ccaa"].map({c: g for g, cs in GRUPOS_CCAA.items() for c in cs})
sin_grupo = sorted(df_nan_clean.loc[grupo.isna(), "ccaa"].unique())
print(f"CCAA sin grupo (se conservan siempre): {sin_grupo}")
mes, dia = df_nan_clean["month"], df_nan_clean["acq_date"].dt.day
ini_mes = grupo.map({g: v[0][0] for g, v in VENTANAS.items()})
ini_dia = grupo.map({g: v[0][1] for g, v in VENTANAS.items()})
fin_mes = grupo.map({g: v[1][0] for g, v in VENTANAS.items()})
fin_dia = grupo.map({g: v[1][1] for g, v in VENTANAS.items()})
dentro = (
    (mes > ini_mes) | ((mes == ini_mes) & (dia >= ini_dia))
    | (mes < fin_mes) | ((mes == fin_mes) & (dia <= fin_dia))
    | grupo.isna()
)
sospecha = dentro & df_nan_clean["clc_class"].isin(CLC_QUEMA)

print(f"Total: {len(df_nan_clean)} | dentro: {int(dentro.sum())} | fuera: {int((~dentro).sum())} | sospechosas: {int(sospecha.sum())}")
print(pd.crosstab(grupo.fillna("sin-grupo"), sospecha))
frp_ok, frp_sosp = df_nan_clean.loc[~sospecha, "frp"].median(), df_nan_clean.loc[sospecha, "frp"].median()
print(f"FRP mediana (conservadas vs sospechosas): {frp_ok} vs {frp_sosp}")
print("Mes de las sospechosas:")
print(df_nan_clean.loc[sospecha, "month"].value_counts().sort_index().to_string())

if int(sospecha.sum()) and not (frp_sosp < frp_ok):
    raise ValueError("La heurística no separa por FRP: revisar VENTANAS/CLC_QUEMA antes de filtrar")

quemas = df_nan_clean[sospecha].copy()
df_nan_clean = df_nan_clean[~sospecha].copy()
save_csv(quemas, "data/processed/expanded/quemas_agricolas_candidatas.csv")
save_csv(df_nan_clean, "data/processed/expanded/wildfire_data_cleaned.csv")
print(f"Conservadas: {df_nan_clean.shape} -> wildfire_data_cleaned.csv | Apartadas: {quemas.shape} -> quemas_agricolas_candidatas.csv")
